# Empirical Categorical Landscape

Beyond sequences, `GraphFLA` handles arbitrary combinatorial spaces. Here we use a chemistry screen from:

Thomas M McGuire *et al.*, "Chemical recycling of polyesters and polycarbonates: why is Zinc(II) such an effective depolymerization catalyst?" *J. Am. Chem. Soc.* (2025). 8 metal catalysts × 4 polyester substrates = 32-cell screen; fitness is the log₁₀ depolymerization rate constant.

Both axes are nominal (no inherent ordering), so we declare them as `categorical`.

## 1. Load the data

In [1]:
import pandas as pd

df = pd.read_csv('../data/Chemistry/Depoly/depoly_rates.csv')
df[['metal', 'polymer', 'log10_k_rate']].head()

,metal,polymer,log10_k_rate
0,Ba,PCL,-3.954884
1,Bi,PCL,-3.878191
2,Ca,PCL,-3.709932
3,Co,PCL,-3.036818
4,Mg,PCL,-3.406790


## 2. Build the landscape

In [2]:
from graphfla.landscape import Landscape

X = df[['metal', 'polymer']]
f = df['log10_k_rate']

landscape = Landscape(maximize=True)
landscape.build_from_data(
    X, f,
    data_types={'metal': 'categorical', 'polymer': 'categorical'},
    verbose=True,
)

Building Landscape from data...
 - Validating data types dictionary...
   - Data types dictionary validation successful.
 - Handling missing values and duplicates...
 - Checking for missing values...
 - Handling duplicate configurations...
   - No duplicate configurations found.
Preparing data for landscape construction (encoding variables)...
Constructing landscape graph...
 - Auto-selected 'pairwise' neighborhood strategy.
 - Finding Hamming-1 neighbors for 32 configurations (masked-position grouping)...
 - Found 160 neighbor pairs within edit distance 1.
 - Identified 160 improving connections.
 - Constructing graph object...
 - Adding node attributes (fitness, etc.)...
Calculating landscape properties...
 - Calculating network metrics (degrees)...
 - Determining local optima...
   - Found 1 local optima.
 - Determining global optimum...
   - Global optimum found at index 23 with fitness -1.8110.
Landscape built successfully.



Landscape(kind='default', maximize=True)

## 3. Basic information

In [3]:
print(f"Size: {len(landscape)}")
print(f"Dimension: {landscape.n_vars}")
print(f"Number of local optima: {landscape.n_lo}")
print(f"Global optimum index: {landscape.go_index}")
print("Global optimum information:")
print(landscape[landscape.go_index])

Size: 32
Dimension: 2
Number of local optima: 1
Global optimum index: 23
Global optimum information:
{'metal': 'Zn', 'polymer': 'PHL', 'fitness': np.float64(-1.811028079107334), 'in_degree': 10, 'out_degree': 0, 'is_lo': True}


## 4. Landscape features

In [4]:
from graphfla.analysis import (
    local_optima_ratio, autocorrelation, fdc,
)

print(f"lo_ratio:        {local_optima_ratio(landscape):.4f}")
print(f"autocorrelation: {autocorrelation(landscape):.4f}")
print(f"fdc:             {fdc(landscape):.4f}")


 - Calculating distances to global optimum using mixed_distance...
   - Distances to GO calculated and added as node attribute 'dist_go'.


lo_ratio:        0.0312
autocorrelation: 0.1936
fdc:             -0.4591


### The whole feature panel in one call

`analysis.profile()` runs every whole-landscape metric and returns a tidy `pandas` Series — the computed-metric companion to `landscape.describe()`. Pass a list of landscapes to get a `DataFrame`, one row each, for side-by-side comparison.

In [5]:
from graphfla import analysis

# one call -> a pandas Series of every whole-landscape metric
analysis.profile(landscape)

Output()

✓ fitness_distribution                0.0s

✓ local_optima_ratio                  0.0s

✓ gradient_intensity                  0.0s

✓ autocorrelation                     0.0s

...

✓ increasing_costs_index              0.0s

✓ classify_epistasis                  0.0s

✓ extradimensional_bypass             0.0s

✓ profiled 22 features in 2.7s

fitness.skewness                      0.982960
fitness.kurtosis                      2.810183
fitness.cv                            0.201811
fitness.quartile_coefficient          0.251514
fitness.median_mean_ratio             1.098375
fitness.relative_range                0.699588
fitness.cauchy_loc                   -3.910249
local_optima_ratio                    0.031250
gradient_intensity                   -0.185523
autocorrelation                       0.200439
r_s_ratio                             0.385126
neutrality                            0.025000
evolvability_enhancing_mutations      0.787500
fdc                                  -0.459120
basin_fitness_correlation                  NaN
neighbor_fitness_correlation          0.805387
fitness_flattening_index             -0.904762
global_optima_accessibility           1.000000
mean_path_length_to_global_optimum    1.625000
mean_distance_to_global_optimum       1.625000
gamma                                 0.840635
gamma_star   

## 5. Per-mutation effects

In [6]:
from graphfla.analysis import all_mutation_effects

all_mutation_effects(landscape)

,mutation_from,mutation_to,median_abs_effect,mean_effect,p_value,significant
0,Ba,Bi,0.165814,0.090951,0.312500,False
1,Ba,Ca,0.233479,-0.259741,1.000000,False
2,Ba,Co,1.532034,-1.209213,1.000000,False
3,Ba,Mg,0.595320,-0.531399,1.000000,False
4,Ba,Sn,0.273383,0.262308,0.062500,False
5,Ba,Y,0.170358,0.087964,0.687500,False
6,Ba,Zn,2.353870,-1.660545,1.000000,False
7,Bi,Ca,0.323597,-0.350692,1.000000,False
8,Bi,Co,1.732680,-1.300164,1.000000,False
9,Bi,Mg,0.685438,-0.622350,1.000000,False
